# Current Price Prediction Without Leakage

This notebook replaces the previous `current_price` experiment with a leakage-audited workflow.

## Leakage audit outcome

The prior winner was invalid because it used fields that directly encode the target:

- `original_price`
- `discount_text`
- `current_price_text`
- `raw_text`
- `raw_payload['prices']`
- any badge or text field that contains literal prices

### Concrete evidence

- `raw_text` is built from `brand`, `title`, `seller`, `current_price_text`, `original_price_text`, and `discount_text`.
- On `8,315` rows, the formula `original_price * (1 - discount_pct)` reconstructs `current_price` with MAE `0.2223` and exact match rate `0.3898`.
- Therefore, the earlier RMSE around `45.89` was not a valid estimate for a real predictive setup.

## Valid experiment scope

This notebook keeps only leakage-free inputs:

- search context: `query`, audience/root derived from query, page and rank position
- product metadata: `brand`, `seller`, `sellerId`, `GSCCategoryId`, `availability`, provider flag
- weak behavioral signals: `rating`, `review_count`, `sponsored`
- clean description signal: `title` transformed with `TF-IDF + SVD`
- image signal: first image ResNet18 embeddings with PyTorch on `mps`

## Recorded leaderboard from the corrected run

| experiment | family | scope | RMSE mean | RMSE std | MAE mean | R2 mean |
|---|---|---|---:|---:|---:|---:|
| `cb_leakfree_title_tfidf_deeper` | catboost | full dataset | 54.6126 | 4.1652 | 27.1999 | 0.7822 |
| `cb_leakfree_title_tfidf` | catboost | full dataset | 57.0764 | 4.4171 | 28.8252 | 0.7621 |
| `cb_leakfree_structured` | catboost | full dataset | 62.4258 | 4.6621 | 32.0394 | 0.7156 |
| `cb_subset_leakfree_title_tfidf` | catboost | image sample of 420 unique SKUs | 76.0001 | 18.4050 | 44.9848 | 0.4935 |
| `cb_subset_leakfree_title_tfidf_image` | catboost + pytorch image embeddings | image sample of 420 unique SKUs | 76.2439 | 18.4545 | 45.1092 | 0.4901 |

## Winner model

```text
Winner model: cb_leakfree_title_tfidf_deeper
Family: catboost
Primary metric: rmse = 54.6126
Secondary metrics: mae = 27.1999, r2 = 0.7822
CV detail: 50.8708, 50.5739, 52.3288, 59.1090, 60.1803
Key features: query/category, seller-brand metadata, ranking signals, rating-review signals, title-derived counts, TF-IDF title components
Key params: depth=8, learning_rate=0.05, iterations=380, GroupKFold(5), log1p target
Notebook: notebook/20260312-current-price-no-leakage.ipynb
Next iteration: collect richer PDP descriptions/specifications and retry the image hybrid on a stronger vision-text fusion setup
```


## Reproducibility note

The project environment does not include the data science stack by default. The corrected run used ephemeral dependencies via `uv run --with ...`.

```bash
uv run   --with numpy   --with pandas   --with scikit-learn   --with catboost   --with torch   --with torchvision   --with pillow   --with httpx   jupyter lab
```


In [ ]:
from __future__ import annotations

import io
import json
import math
import sqlite3
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import httpx
import numpy as np
import pandas as pd
import torch
from PIL import Image
from catboost import CatBoostRegressor, Pool
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, KFold
from torchvision.models import ResNet18_Weights, resnet18

DB_PATH = Path('../instance/pricing_prediction.db') if Path('../instance/pricing_prediction.db').exists() else Path('instance/pricing_prediction.db')
RANDOM_STATE = 42
PRIMARY_METRIC = 'rmse'
SECONDARY_METRICS = ['mae', 'r2']
STRUCTURED_SPLITS = 5
IMAGE_SPLITS = 5
IMAGE_SAMPLE_PER_QUERY = 70
RUN_IMAGE_EXPERIMENT = True

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)
print(DB_PATH)


## Leakage verification

This cell re-checks the exact leakage claim on the raw database.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    leakage_df = pd.read_sql_query(
        '''
        SELECT current_price, original_price, discount_text, raw_text, current_price_text, original_price_text
        FROM product_snapshots
        WHERE current_price IS NOT NULL
        ''',
        conn,
    )

formula_rows = leakage_df.dropna(subset=['original_price', 'discount_text']).copy()
formula_rows['discount_pct'] = (
    formula_rows['discount_text']
    .str.extract(r'(-?\d+(?:\.\d+)?)')[0]
    .astype(float)
    .abs()
    / 100.0
)
formula_rows = formula_rows.dropna(subset=['discount_pct'])
formula_rows['formula_price'] = (formula_rows['original_price'] * (1.0 - formula_rows['discount_pct'])).round(2)
formula_rows['abs_error'] = (formula_rows['formula_price'] - formula_rows['current_price']).abs()

leakage_summary = {
    'rows_with_original_and_discount': len(formula_rows),
    'formula_mae': round(float(formula_rows['abs_error'].mean()), 4),
    'formula_exact_match_rate': round(float((formula_rows['abs_error'] < 0.011).mean()), 4),
    'raw_text_example': leakage_df['raw_text'].iloc[0],
}
leakage_summary


## Data loading and leakage-free feature set

The SQL query intentionally excludes all price-derived fields from the feature set. `raw_payload` is only used to derive safe metadata such as category ID, seller ID, availability, media count, and flags that do not expose price.


In [ ]:
FULL_SQL = '''
WITH image_agg AS (
    SELECT
        sku_id,
        COUNT(*) AS image_count,
        MAX(CASE WHEN position = 1 THEN image_url END) AS first_image_url
    FROM product_images
    GROUP BY sku_id
)
SELECT
    ps.id,
    ps.sku_id,
    ps.query,
    ps.page_number,
    ps.position,
    CAST(ps.current_price AS REAL) AS current_price,
    CAST(ps.rating AS REAL) AS rating,
    ps.review_count,
    COALESCE(NULLIF(ps.seller, ''), NULLIF(p.seller, ''), 'unknown') AS seller,
    CASE WHEN ps.sponsored THEN 1 ELSE 0 END AS sponsored,
    COALESCE(NULLIF(p.brand, ''), 'unknown') AS brand,
    p.title,
    p.source_domain,
    ps.raw_payload,
    img.image_count,
    img.first_image_url
FROM product_snapshots ps
JOIN products p ON p.sku_id = ps.sku_id
LEFT JOIN image_agg img ON img.sku_id = ps.sku_id
WHERE ps.current_price IS NOT NULL
'''

with sqlite3.connect(DB_PATH) as conn:
    full_df = pd.read_sql_query(FULL_SQL, conn)

summary = {
    'rows': len(full_df),
    'distinct_skus': full_df['sku_id'].nunique(),
    'queries': full_df['query'].nunique(),
    'rows_with_images': int(full_df['first_image_url'].notna().sum()),
}
summary


In [ ]:
def query_root(value: str) -> str:
    text = str(value).lower()
    if 'zapatos' in text:
        return 'zapatos'
    if 'ropa' in text:
        return 'ropa'
    return 'other'


def query_audience(value: str) -> str:
    text = str(value).lower()
    if 'niñ' in text or 'nino' in text or 'niño' in text:
        return 'kids'
    if 'mujer' in text:
        return 'women'
    if 'hombre' in text:
        return 'men'
    return 'other'


def image_namespace(value: object) -> str:
    if not isinstance(value, str):
        return 'missing'
    parts = value.split('/')
    return parts[3] if len(parts) > 3 else 'missing'


def availability_bucket(payload: dict) -> str:
    availability = payload.get('availability') or {}
    if not isinstance(availability, dict):
        return 'unknown'
    if availability.get('internationalShipping'):
        return 'international'
    return 'domestic'


def payload_features(payload: dict) -> dict[str, object]:
    media_urls = payload.get('mediaUrls') or []
    return {
        'gsc_category_id': str(payload.get('GSCCategoryId') or 'unknown'),
        'provider_name': str(payload.get('providerName') or 'unknown'),
        'seller_id': str(payload.get('sellerId') or 'unknown'),
        'availability_bucket': availability_bucket(payload),
        'payload_is_best_seller': int(bool(payload.get('isBestSeller'))),
        'payload_is_frequent_product': int(bool(payload.get('isFrequentProduct'))),
        'multi_badge_count': len(payload.get('multipurposeBadges') or []),
        'media_url_count': len(media_urls),
    }


def build_feature_frame(df: pd.DataFrame) -> pd.DataFrame:
    parsed = df['raw_payload'].map(json.loads).map(payload_features).apply(pd.Series)
    frame = pd.concat([df.drop(columns=['raw_payload']), parsed], axis=1)
    frame['query_root'] = frame['query'].map(query_root)
    frame['query_audience'] = frame['query'].map(query_audience)
    frame['image_namespace'] = frame['first_image_url'].map(image_namespace)
    frame['rating_missing'] = frame['rating'].isna().astype(int)
    frame['rating'] = frame['rating'].fillna(frame['rating'].median())
    frame['review_count'] = frame['review_count'].fillna(0)
    frame['review_count_log1p'] = np.log1p(frame['review_count'])
    frame['image_count'] = frame['image_count'].fillna(0).astype(int)
    frame['title'] = frame['title'].fillna('')
    frame['title_word_count'] = frame['title'].str.split().str.len().fillna(0)
    frame['title_char_count'] = frame['title'].str.len().fillna(0)
    frame['title_digit_count'] = frame['title'].str.count(r'\d')
    frame['title_has_pack'] = frame['title'].str.contains(r'pack|set|kit', case=False, regex=True).astype(int)
    frame['title_has_kids_token'] = frame['title'].str.contains(r'niñ|nino|niño|kid|junior', case=False, regex=True).astype(int)
    frame['title_has_sport_token'] = frame['title'].str.contains(r'deportivo|running|trek|outdoor|gym|sport', case=False, regex=True).astype(int)
    frame['brand_in_title'] = frame.apply(lambda row: int(str(row['brand']).lower() in str(row['title']).lower()), axis=1)
    frame['rank_position'] = (frame['page_number'] - 1) * 48 + frame['position']
    frame['log_target'] = np.log1p(frame['current_price'])
    frame['title_text'] = (frame['brand'].fillna('') + ' ' + frame['query'].fillna('') + ' ' + frame['title'].fillna('')).str.lower()
    return frame

feature_df = build_feature_frame(full_df)
feature_df.head(3)


## Leakage-free CatBoost experiments

`GroupKFold` is kept because some products appear in multiple snapshots. This prevents the same `sku_id` from landing in both train and validation.

Text is included through a leakage-free `TF-IDF + SVD` representation of `title_text`, not through price-bearing raw text.


In [ ]:
BASE_FEATURES = [
    'query', 'query_root', 'query_audience', 'brand', 'seller', 'seller_id', 'source_domain',
    'gsc_category_id', 'provider_name', 'availability_bucket', 'image_namespace',
    'page_number', 'position', 'rank_position', 'rating', 'rating_missing', 'review_count_log1p',
    'image_count', 'media_url_count', 'title_word_count', 'title_char_count', 'title_digit_count',
    'title_has_pack', 'title_has_kids_token', 'title_has_sport_token', 'brand_in_title', 'sponsored',
    'payload_is_best_seller', 'payload_is_frequent_product', 'multi_badge_count'
]
CAT_FEATURES = [
    'query', 'query_root', 'query_audience', 'brand', 'seller', 'seller_id', 'source_domain',
    'gsc_category_id', 'provider_name', 'availability_bucket', 'image_namespace'
]


def add_text_svd(train_df: pd.DataFrame, valid_df: pd.DataFrame, n_components: int = 48):
    vectorizer = TfidfVectorizer(max_features=4000, ngram_range=(1, 2), min_df=3)
    train_matrix = vectorizer.fit_transform(train_df['title_text'])
    valid_matrix = vectorizer.transform(valid_df['title_text'])
    svd = TruncatedSVD(n_components=min(n_components, train_matrix.shape[1] - 1), random_state=RANDOM_STATE)
    train_svd = svd.fit_transform(train_matrix)
    valid_svd = svd.transform(valid_matrix)
    cols = [f'title_svd_{i + 1}' for i in range(train_svd.shape[1])]
    train_aug = pd.concat([train_df.reset_index(drop=True), pd.DataFrame(train_svd, columns=cols)], axis=1)
    valid_aug = pd.concat([valid_df.reset_index(drop=True), pd.DataFrame(valid_svd, columns=cols)], axis=1)
    return train_aug, valid_aug, cols


LEAKFREE_EXPERIMENTS = {
    'cb_leakfree_structured': {'use_text_svd': False, 'params': {'depth': 7, 'learning_rate': 0.06, 'iterations': 320}},
    'cb_leakfree_title_tfidf': {'use_text_svd': True, 'params': {'depth': 7, 'learning_rate': 0.05, 'iterations': 320}},
    'cb_leakfree_title_tfidf_deeper': {'use_text_svd': True, 'params': {'depth': 8, 'learning_rate': 0.05, 'iterations': 380}},
}


def evaluate_leakfree(frame: pd.DataFrame, experiments: dict[str, dict]) -> pd.DataFrame:
    splitter = GroupKFold(n_splits=STRUCTURED_SPLITS)
    rows = []
    for name, cfg in experiments.items():
        fold_rmse = []
        fold_mae = []
        fold_r2 = []
        for train_idx, valid_idx in splitter.split(frame, groups=frame['sku_id']):
            train_df = frame.iloc[train_idx].copy()
            valid_df = frame.iloc[valid_idx].copy()
            features = list(BASE_FEATURES)
            if cfg['use_text_svd']:
                train_df, valid_df, text_cols = add_text_svd(train_df, valid_df)
                features += text_cols
            train_pool = Pool(train_df[features], label=train_df['log_target'], cat_features=CAT_FEATURES)
            valid_pool = Pool(valid_df[features], label=valid_df['log_target'], cat_features=CAT_FEATURES)
            model = CatBoostRegressor(
                loss_function='RMSE',
                eval_metric='RMSE',
                random_seed=RANDOM_STATE,
                verbose=False,
                allow_writing_files=False,
                od_type='Iter',
                od_wait=40,
                **cfg['params'],
            )
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
            pred = np.expm1(model.predict(valid_pool))
            actual = np.expm1(valid_df['log_target'])
            fold_rmse.append(math.sqrt(mean_squared_error(actual, pred)))
            fold_mae.append(mean_absolute_error(actual, pred))
            fold_r2.append(r2_score(actual, pred))
        rows.append({
            'name': name,
            'family': 'catboost',
            'cv_mean_rmse': float(np.mean(fold_rmse)),
            'cv_std_rmse': float(np.std(fold_rmse)),
            'cv_mean_mae': float(np.mean(fold_mae)),
            'cv_std_mae': float(np.std(fold_mae)),
            'cv_mean_r2': float(np.mean(fold_r2)),
            'cv_std_r2': float(np.std(fold_r2)),
            'fold_rmse': [round(float(x), 4) for x in fold_rmse],
        })
    return pd.DataFrame(rows).sort_values(['cv_mean_rmse', 'cv_std_rmse']).reset_index(drop=True)

leakfree_leaderboard = evaluate_leakfree(feature_df, LEAKFREE_EXPERIMENTS)
leakfree_leaderboard


## Image branch with PyTorch MPS

The image experiment uses the first image per sampled unique SKU, extracts ResNet18 embeddings on `mps`, reduces them with PCA, and compares a text-only subset against a text+image hybrid.


In [ ]:
def build_image_sample(frame: pd.DataFrame, sample_per_query: int = IMAGE_SAMPLE_PER_QUERY) -> pd.DataFrame:
    parts = []
    for _, group in frame.groupby('query'):
        unique_group = group.sort_values('sku_id').drop_duplicates('sku_id')
        parts.append(unique_group.sample(n=min(len(unique_group), sample_per_query), random_state=RANDOM_STATE))
    return pd.concat(parts, ignore_index=True)


def fetch_image(url: str, client: httpx.Client):
    try:
        response = client.get(url)
        response.raise_for_status()
        return Image.open(io.BytesIO(response.content)).convert('RGB')
    except Exception:
        return None


def compute_image_embeddings(frame: pd.DataFrame):
    weights = ResNet18_Weights.DEFAULT
    preprocess = weights.transforms()
    device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
    backbone = resnet18(weights=weights)
    backbone.fc = torch.nn.Identity()
    backbone.eval()
    backbone.to(device)

    client = httpx.Client(timeout=20.0, follow_redirects=True, headers={'User-Agent': 'pricing-prediction-experiment/1.0'})
    images = {}
    with ThreadPoolExecutor(max_workers=12) as pool:
        future_map = {pool.submit(fetch_image, row.first_image_url, client): idx for idx, row in frame.iterrows()}
        for future in as_completed(future_map):
            idx = future_map[future]
            image = future.result()
            if image is not None:
                images[idx] = image
    client.close()

    valid_indices = sorted(images)
    valid_frame = frame.iloc[valid_indices].reset_index(drop=True)
    stack = torch.stack([preprocess(images[idx]) for idx in valid_indices])
    embeddings = []
    with torch.inference_mode():
        for start in range(0, len(stack), 32):
            batch = stack[start:start + 32].to(device)
            embeddings.append(backbone(batch).detach().cpu().numpy())
    embedding_matrix = np.vstack(embeddings)
    pca = PCA(n_components=min(24, embedding_matrix.shape[0], embedding_matrix.shape[1]), random_state=RANDOM_STATE)
    pcs = pca.fit_transform(embedding_matrix)
    image_cols = []
    for idx in range(pcs.shape[1]):
        col = f'image_pc_{idx + 1}'
        valid_frame[col] = pcs[:, idx]
        image_cols.append(col)
    return valid_frame, image_cols, str(device)


def evaluate_image_branch(frame: pd.DataFrame, image_cols: list[str]) -> pd.DataFrame:
    experiments = {
        'cb_subset_leakfree_title_tfidf': {'use_image': False},
        'cb_subset_leakfree_title_tfidf_image': {'use_image': True},
    }
    rows = []
    splitter = KFold(n_splits=IMAGE_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    for name, cfg in experiments.items():
        fold_rmse = []
        fold_mae = []
        fold_r2 = []
        for train_idx, valid_idx in splitter.split(frame):
            train_df = frame.iloc[train_idx].copy()
            valid_df = frame.iloc[valid_idx].copy()
            train_df, valid_df, text_cols = add_text_svd(train_df, valid_df, n_components=32)
            features = list(BASE_FEATURES) + text_cols
            if cfg['use_image']:
                features += image_cols
            train_pool = Pool(train_df[features], label=train_df['log_target'], cat_features=CAT_FEATURES)
            valid_pool = Pool(valid_df[features], label=valid_df['log_target'], cat_features=CAT_FEATURES)
            model = CatBoostRegressor(
                loss_function='RMSE',
                eval_metric='RMSE',
                random_seed=RANDOM_STATE,
                verbose=False,
                allow_writing_files=False,
                od_type='Iter',
                od_wait=30,
                depth=7,
                learning_rate=0.05,
                iterations=260,
            )
            model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
            pred = np.expm1(model.predict(valid_pool))
            actual = np.expm1(valid_df['log_target'])
            fold_rmse.append(math.sqrt(mean_squared_error(actual, pred)))
            fold_mae.append(mean_absolute_error(actual, pred))
            fold_r2.append(r2_score(actual, pred))
        rows.append({
            'name': name,
            'family': 'catboost' if not cfg['use_image'] else 'catboost + pytorch',
            'cv_mean_rmse': float(np.mean(fold_rmse)),
            'cv_std_rmse': float(np.std(fold_rmse)),
            'cv_mean_mae': float(np.mean(fold_mae)),
            'cv_std_mae': float(np.std(fold_mae)),
            'cv_mean_r2': float(np.mean(fold_r2)),
            'cv_std_r2': float(np.std(fold_r2)),
            'fold_rmse': [round(float(x), 4) for x in fold_rmse],
        })
    return pd.DataFrame(rows).sort_values(['cv_mean_rmse', 'cv_std_rmse']).reset_index(drop=True)

if RUN_IMAGE_EXPERIMENT:
    image_sample = build_image_sample(feature_df)
    image_frame, image_cols, image_device = compute_image_embeddings(image_sample)
    image_leaderboard = evaluate_image_branch(image_frame, image_cols)
    print({'sample_rows': len(image_sample), 'downloaded_rows': len(image_frame), 'device': image_device})
    image_leaderboard
else:
    print('Set RUN_IMAGE_EXPERIMENT = True to rerun the image branch.')


## Recorded corrected results

These values come from the rerun after removing leakage.


In [ ]:
recorded_results = pd.DataFrame([
    {
        'name': 'cb_leakfree_structured',
        'family': 'catboost',
        'scope': 'full_dataset',
        'cv_mean_rmse': 62.4258,
        'cv_std_rmse': 4.6621,
        'cv_mean_mae': 32.0394,
        'cv_std_mae': 0.7194,
        'cv_mean_r2': 0.7156,
        'cv_std_r2': 0.0308,
        'fold_rmse': [58.0251, 57.0590, 61.3509, 68.3803, 67.3139],
        'notes': 'Leakage-free structured baseline.',
    },
    {
        'name': 'cb_leakfree_title_tfidf',
        'family': 'catboost',
        'scope': 'full_dataset',
        'cv_mean_rmse': 57.0764,
        'cv_std_rmse': 4.4171,
        'cv_mean_mae': 28.8252,
        'cv_std_mae': 0.8932,
        'cv_mean_r2': 0.7621,
        'cv_std_r2': 0.0277,
        'fold_rmse': [52.2128, 53.2113, 55.2567, 62.4404, 62.2606],
        'notes': 'Title TF-IDF adds meaningful signal without leakage.',
    },
    {
        'name': 'cb_leakfree_title_tfidf_deeper',
        'family': 'catboost',
        'scope': 'full_dataset',
        'cv_mean_rmse': 54.6126,
        'cv_std_rmse': 4.1652,
        'cv_mean_mae': 27.1999,
        'cv_std_mae': 0.8042,
        'cv_mean_r2': 0.7822,
        'cv_std_r2': 0.0246,
        'fold_rmse': [50.8708, 50.5739, 52.3288, 59.1090, 60.1803],
        'notes': 'Corrected winner.',
    },
    {
        'name': 'cb_subset_leakfree_title_tfidf',
        'family': 'catboost',
        'scope': 'image_sample_420',
        'cv_mean_rmse': 76.0001,
        'cv_std_rmse': 18.4050,
        'cv_mean_mae': 44.9848,
        'cv_std_mae': 7.6012,
        'cv_mean_r2': 0.4935,
        'cv_std_r2': 0.1070,
        'fold_rmse': [56.6117, 72.4392, 73.4158, 66.7161, 110.8175],
        'notes': 'Leakage-free title baseline on the image subset.',
    },
    {
        'name': 'cb_subset_leakfree_title_tfidf_image',
        'family': 'catboost + pytorch',
        'scope': 'image_sample_420',
        'cv_mean_rmse': 76.2439,
        'cv_std_rmse': 18.4545,
        'cv_mean_mae': 45.1092,
        'cv_std_mae': 7.6712,
        'cv_mean_r2': 0.4901,
        'cv_std_r2': 0.1109,
        'fold_rmse': [56.0057, 74.6002, 76.9111, 63.7745, 109.9280],
        'notes': 'Current image fusion did not improve the leakage-free title baseline.',
    },
]).sort_values(['cv_mean_rmse', 'cv_std_rmse']).reset_index(drop=True)

recorded_results


## Takeaways

- The earlier experiment had real leakage and should not be used as the production benchmark.
- Once price-derived fields are removed, the best CatBoost model lands at `RMSE 54.61` and `MAE 27.20`.
- Clean description signal from `title` materially helps.
- The current image branch does not beat the non-image title model yet.
- The dataset has very limited extra description text in `raw_payload`; `topSpecifications` is almost always empty, so richer PDP data would likely be the biggest next gain.
